In [ ]:
import sys
import os
import pandas as pd

sys.path.insert(0, os.path.abspath('../..'))
from vpei.models import MODELS, MODELS_WITH_REASON_OFF

In [ ]:
arena_df = pd.read_csv('llm_arena_ratings.csv')
print(f'Arena ratings: {len(arena_df)} models')
arena_df.head()

In [ ]:
# Suffixes indicating reasoning is active in the arena entry — exclude these
# when mapping models that run with reasoning disabled.
REASONING_SUFFIXES = [
    '-high',          # high reasoning effort (OpenAI naming)
    '-thinking',      # extended thinking
    '-reasoning',     # reasoning mode
    '-thinking-32k',  # thinking with token budget
    '-thinking-16k',
]
REASONING_SUBSTRINGS = [
    '(thinking',      # e.g. 'gemini-3-flash (thinking-minimal)'
]

def is_reasoning_active(name):
    if any(name.endswith(s) for s in REASONING_SUFFIXES):
        return True
    if any(s in name for s in REASONING_SUBSTRINGS):
        return True
    return False

non_reasoning_arena = arena_df[~arena_df['model_name'].apply(is_reasoning_active)]
print(f'Non-reasoning arena entries: {len(non_reasoning_arena)} (filtered out {len(arena_df) - len(non_reasoning_arena)})')
non_reasoning_arena.head(20)

In [ ]:
# Manual mapping: experiment model name -> arena model name.
# Only map to non-reasoning arena entries.
# Models where only a -high variant exists in arena are left unmapped (NaN).
ARENA_MODEL_MAPPING = {
    # OpenAI
    'gpt-5.4':        'gpt-5.4',                  # gpt-5.4 (non-high) exists
    # gpt-5.4-mini: only gpt-5.4-mini-high in arena -> NaN
    # gpt-5.4-nano: only gpt-5.4-nano-high in arena -> NaN
    'gpt-5':          'gpt-5-chat',                # gpt-5-chat is the non-high variant
    # gpt-5-mini: only gpt-5-mini-high in arena -> NaN
    # gpt-5-nano: only gpt-5-nano-high in arena -> NaN
    'gpt-4.1':        'gpt-4.1-2025-04-14',
    'gpt-4.1-mini':   'gpt-4.1-mini-2025-04-14',
    'gpt-4.1-nano':   'gpt-4.1-nano-2025-04-14',
    'gpt-4o':         'gpt-4o-2024-08-06',
    'gpt-4o-mini':    'gpt-4o-mini-2024-07-18',
    'gpt-3.5-turbo':  'gpt-3.5-turbo-0125',
    # xAI
    'grok-4.20-non-reasoning':    'grok-4.20-beta1',   # non-reasoning arena entry
    'grok-4-1-fast-non-reasoning': 'grok-4.1',
    # Anthropic
    'claude-sonnet-4-6':          'claude-sonnet-4-6',
    'claude-haiku-4-5-20251001':  'claude-haiku-4-5-20251001',
    # Google
    'gemini-3-flash-preview':           'gemini-3-flash',
    'gemini-3.1-flash-lite-preview':    'gemini-3.1-flash-lite-preview',
    # TogetherAI
    'Qwen/Qwen3.5-397B-A17B':                                              'qwen3.5-397b-a17b',
    'moonshotai/Kimi-K2.5':                                                'kimi-k2.5-instant',     # instant = non-thinking
    'deepseek-ai/DeepSeek-V3.1':                                           'deepseek-v3.1',
    'meta-llama/Llama-3.3-70B-Instruct-Turbo':                            'llama-3.3-70b-instruct',
    'openai/gpt-oss-120b':                                                 'gpt-oss-120b',
    'zai-org/GLM-5.1':                                                     'glm-5.1',
    'drozado/meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8-4e57e3dc': 'llama-4-maverick-17b-128e-instruct',
    'drozado/mistralai/Mixtral-8x7B-Instruct-v0.1-11d06fa6':              'mixtral-8x7b-instruct-v0.1',
    'drozado/meta-llama/Meta-Llama-3-70B-Instruct-Turbo-0019830d':        'llama-3-70b-instruct',
    'drozado/google/gemma-2-9b-it-c190a2df':                              'gemma-2-9b-it',
    'drozado/meta-llama/Meta-Llama-3.1-8B-Instruct-Turbo-1030ae43':       'llama-3.1-8b-instruct',
    'drozado/mistralai/Mixtral-8x22B-Instruct-v0.1-99187f2a':             'mixtral-8x22b-instruct-v0.1',
}

In [ ]:
# Verify all mapped arena names exist in the non-reasoning set
arena_names = set(non_reasoning_arena['model_name'])
missing = {k: v for k, v in ARENA_MODEL_MAPPING.items() if v not in arena_names}
if missing:
    print('WARNING - mapped arena names not found in non-reasoning entries:')
    for k, v in missing.items():
        print(f'  {k} -> {v}')
else:
    print('All mapped arena names verified as non-reasoning entries.')

In [ ]:
arena_ratings_lookup = arena_df.set_index('model_name')['rating'].to_dict()

rows = []
for model_name in MODELS_WITH_REASON_OFF:
    long_name = MODELS[model_name]['long_name']
    arena_name = ARENA_MODEL_MAPPING.get(model_name)
    if arena_name is None:
        continue
    rating = arena_ratings_lookup.get(arena_name)
    if rating is None:
        continue
    rows.append({'model_name': model_name, 'long_name': long_name, 'arena_rating': rating})

result_df = pd.DataFrame(rows)
result_df

In [ ]:
skipped = [m for m in MODELS_WITH_REASON_OFF if m not in result_df['model_name'].values]
print(f'{len(skipped)} models excluded (no non-reasoning arena entry):')
for m in skipped:
    print(f'  {m}')

In [ ]:
result_df.to_csv('experiment_models_to_llm_arena_rating.csv', index=False)
print('Saved experiment_models_to_llm_arena_rating.csv')